## Twitter Sentimental Analysis
The following project is about analyzing the sentiments of tweets on social networking website
‘Twitter’. The dataset for this project is scraped from Twitter. It contains 1,600,000 tweets
extracted using Twitter API. It is a labeled dataset with tweets annotated with the sentiment (0 =
negative, 2 = neutral, 4 = positive).
It contains the following 6 fields:

1. target: the polarity of the tweet (0 = negative, 2 = neutral, 4 = positive)
2. ids: The id of the tweet .
3. date: The date of the tweet (Sat May 16 23:58:44 UTC 2009)
4. flag: The query. If there is no query, then this value is NO_QUERY.
5. user: The user that tweeted
6. text: The text of the tweet.

Design a classification model that correctly predicts the polarity of the tweets provided in the
dataset.

In [1]:
import pandas as pd

#### Loding the CSV File

In [2]:
df=pd.read_csv("twitter_new.csv",names=['target','ids','date','flag','user','text'],on_bad_lines='skip', encoding='latin-1')

In [3]:
df.head(5)

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   target  1600000 non-null  int64 
 1   ids     1600000 non-null  int64 
 2   date    1600000 non-null  object
 3   flag    1600000 non-null  object
 4   user    1600000 non-null  object
 5   text    1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


In [5]:
df.isnull().sum()

target    0
ids       0
date      0
flag      0
user      0
text      0
dtype: int64

In [6]:
df.drop(columns=['ids','date','flag','user'],axis=1,inplace=True)

In [7]:
df.head(5)

,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [8]:
df['text'].duplicated().value_counts()

text
False    1581466
True       18534
Name: count, dtype: int64

In [9]:
df.duplicated().value_counts()

False    1583691
True       16309
Name: count, dtype: int64

In [10]:
df.drop_duplicates(subset=['text'],keep='first',inplace=True)
df.shape

(1581466, 2)

In [11]:
df['text'].duplicated().value_counts()

text
False    1581466
Name: count, dtype: int64

In [12]:
df.target.shape

(1581466,)

In [13]:
df.text.shape

(1581466,)

In [14]:
df['target'].value_counts()

target
4    791281
0    790185
Name: count, dtype: int64

In [15]:
df['target'] = df['target'].replace(4, 1) # Replacing 4 to 1 for classification problem
df['text']=df['text'].str.lower() # if any upper case word converting into lower case

In [16]:
df['target'].value_counts()# its balance dataset

target
1    791281
0    790185
Name: count, dtype: int64

In [21]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
from bs4 import BeautifulSoup

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vinod\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [24]:

URL_PATTERN = re.compile(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?')
SPECIAL_CHAR_PATTERN = re.compile(r'[^a-zA-Z0-9\s-]')  # Optimized whitespace & alphanumeric matching
STOPWORDS_SET = set(stopwords.words('english'))        # Convert list to a set for O(1) lightning-fast lookups

def clean_tweet(text):
    text = str(text)
    
    # Remove HTML tags FIRST while brackets are still intact
    if '<' in text or '&' in text:
        try:  # Quick check so we don't spin up BeautifulSoup unnecessarily
            text = BeautifulSoup(text, 'lxml').get_text()
        except Exception:
            text = BeautifulSoup(text, 'html.parser').get_text()
        
    # Remove URLs
    text = URL_PATTERN.sub('', text)
    
    # Remove Special Characters
    text = SPECIAL_CHAR_PATTERN.sub('', text)
    
    # Remove Stopwords and strip additional spaces simultaneously
    words = [word for word in text.split() if word.lower() not in STOPWORDS_SET]
    
    return " ".join(words)

df['text'] = df['text'].apply(clean_tweet)


In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['text'],df['target'],test_size=0.2,random_state=42)

In [26]:
X_train.shape

(1265172,)

In [27]:
y_train.shape

(1265172,)

###  Transformation of Text Data With Bag of Words (BOW)

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer(ngram_range=(1,2))
X_train_bow=bow.fit_transform(X_train)
X_test_bow=bow.transform(X_test)

###  Transformation of Text Data With TF-IDF (Term Frequency – Inverse Document Frequency)

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, min_df=2,ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

#### Multinomial Naive Bayes with Bow and TF-IDF Tranformed Data

In [54]:
from sklearn.naive_bayes import MultinomialNB

nb_model_bow = MultinomialNB()
nb_model_tfidf = MultinomialNB()
nb_model_bow.fit(X_train_bow, y_train)
nb_model_tfidf.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [55]:
y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)

In [56]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [57]:
y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)
print("Bow Accuracy : ", accuracy_score(y_test,y_pred_bow))
print("TF-IDF Accuracy : ",accuracy_score(y_test,y_pred_tfidf))

Bow Accuracy :  0.7860123808861376
TF-IDF Accuracy :  0.7776056453805636


In [58]:
print('Bow Classification Report')
print(classification_report(y_test, y_pred_bow))

Bow Classification Report
              precision    recall  f1-score   support

           0       0.77      0.82      0.79    157544
           1       0.81      0.75      0.78    158750

    accuracy                           0.79    316294
   macro avg       0.79      0.79      0.79    316294
weighted avg       0.79      0.79      0.79    316294



In [59]:
print('TF-IDF Classification Report')
print(classification_report(y_test, y_pred_tfidf))

TF-IDF Classification Report
              precision    recall  f1-score   support

           0       0.78      0.78      0.78    157544
           1       0.78      0.78      0.78    158750

    accuracy                           0.78    316294
   macro avg       0.78      0.78      0.78    316294
weighted avg       0.78      0.78      0.78    316294



In [36]:
print("confusion_matrix Bow")
print(confusion_matrix(y_test, y_pred_bow))

confusion_matrix Bow
[[129392  28152]
 [ 39531 119219]]


In [37]:
print("confusion_matrix TF-IDF")
print(confusion_matrix(y_test, y_pred_tfidf))

confusion_matrix TF-IDF
[[122828  34716]
 [ 35626 123124]]


BOW: Slightly higher accuracy (0.79) and better recall for class 0.

TF‑IDF: More balanced performance across both classes, but overall accuracy is a bit lower (0.78).

Macro/Weighted averages: Almost identical (0.79 vs 0.78), showing both methods are competitive.

In [38]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    'fit_prior': [True, False] 
}

grid_search = GridSearchCV(estimator=nb, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)

print("Tuning MultinomialNB...")
grid_search.fit(X_train_bow, y_train)

print("Best Parameters Found:", grid_search.best_params_)
print("Best Accuracy Score:", grid_search.best_score_)


Tuning MultinomialNB...
Best Parameters Found: {'alpha': 2.0, 'fit_prior': True}
Best Accuracy Score: 0.7831037993253092


In [39]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    'fit_prior': [True, False]
}


grid_search = GridSearchCV(estimator=nb, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)

print("Tuning MultinomialNB...")
grid_search.fit(X_train_tfidf, y_train)

print("Best Parameters Found:", grid_search.best_params_)
print("Best Accuracy Score:", grid_search.best_score_)

Tuning MultinomialNB...
Best Parameters Found: {'alpha': 5.0, 'fit_prior': False}
Best Accuracy Score: 0.7773614970928854


In [45]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

nb_model_bow = MultinomialNB(alpha= 2.0,fit_prior=True)
nb_model_tfidf = MultinomialNB(alpha= 5.0, fit_prior=False)

nb_model_bow.fit(X_train_bow, y_train)
nb_model_tfidf.fit(X_train_tfidf, y_train)

y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)

y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)
print("Bow Accuracy : ", accuracy_score(y_test,y_pred_bow))
print("TF-IDF Accuracy : ",accuracy_score(y_test,y_pred_tfidf))

print('Bow Classification Report')
print(classification_report(y_test, y_pred_bow))

print('TF-IDF Classification Report')
print(classification_report(y_test, y_pred_tfidf))

Bow Accuracy :  0.787501501767343
TF-IDF Accuracy :  0.7787659582540295
Bow Classification Report
              precision    recall  f1-score   support

           0       0.76      0.83      0.80    157544
           1       0.82      0.75      0.78    158750

    accuracy                           0.79    316294
   macro avg       0.79      0.79      0.79    316294
weighted avg       0.79      0.79      0.79    316294

TF-IDF Classification Report
              precision    recall  f1-score   support

           0       0.78      0.78      0.78    157544
           1       0.78      0.78      0.78    158750

    accuracy                           0.78    316294
   macro avg       0.78      0.78      0.78    316294
weighted avg       0.78      0.78      0.78    316294



In [46]:
print("confusion_matrix Bow")
print(confusion_matrix(y_test, y_pred_bow))
print("confusion_matrix TF-IDF")
print(confusion_matrix(y_test, y_pred_tfidf))

confusion_matrix Bow
[[130669  26875]
 [ 40337 118413]]
confusion_matrix TF-IDF
[[123225  34319]
 [ 35656 123094]]


### Insights: BOW vs TF‑IDF (Twitter Sentiment Analysis)

- **Bag of Words (BOW)**  
  - Achieved slightly higher accuracy (~0.79) compared to TF‑IDF (~0.78).  
  - Stronger recall for class `0` (negative sentiment), meaning it captured negative tweets more effectively.  
  - Frequency‑based representation worked well because tweets are short and context is limited.

- **TF‑IDF**  
  - Produced more balanced precision and recall across both classes.  
  - Down‑weighted common words, but in short texts like tweets, this sometimes reduced useful signal.  
  - Accuracy remained slightly lower (~0.78), showing that raw counts were more effective here.

- **Overall takeaway**  
  - For short, noisy text (tweets), **BOW can outperform TF‑IDF** because raw frequency captures sentiment cues better.  
  - TF‑IDF may shine in longer documents where weighting rare words adds more discriminative power.  
  - Next step: experiment with **n‑grams (bigrams/trigrams)** or embeddings (Word2Vec, GloVe) to capture context beyond single words.


#### Trying With Logistic Regression

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

lr_model = LogisticRegression(max_iter=1000, n_jobs=-1)

print("Training Logistic Regression...")
lr_model.fit(X_train_bow, y_train)

y_pred_lr = lr_model.predict(X_test_bow)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))


Training Logistic Regression...
Logistic Regression Accuracy: 0.8007170543861091

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.78      0.80    157544
           1       0.79      0.82      0.81    158750

    accuracy                           0.80    316294
   macro avg       0.80      0.80      0.80    316294
weighted avg       0.80      0.80      0.80    316294



In [48]:
print("confusion_matrix Bow")
print(confusion_matrix(y_test, y_pred_lr))

confusion_matrix Bow
[[122945  34599]
 [ 28433 130317]]


In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

lr_model = LogisticRegression(max_iter=1000, n_jobs=-1)

print("Training Logistic Regression...")
lr_model.fit(X_train_tfidf, y_train)

y_pred_tfidlr = lr_model.predict(X_test_tfidf)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_tfidlr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tfidlr))
print("\nconfusion_matrix TF-IDF:\n")
print(confusion_matrix(y_test, y_pred_tfidlr))

Training Logistic Regression...
Logistic Regression Accuracy: 0.7953296616439136

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.77      0.79    157544
           1       0.78      0.82      0.80    158750

    accuracy                           0.80    316294
   macro avg       0.80      0.80      0.80    316294
weighted avg       0.80      0.80      0.80    316294


confusion_matrix TF-IDF:

[[122066  35478]
 [ 29258 129492]]


### Insights: Logistic Regression with BOW vs TF‑IDF

- **Bag of Words (BOW)**  
  - Accuracy ~0.795, strong recall for class `1` (positive sentiment).  
  - Captures raw frequency well, but slightly weaker recall for class `0`.

- **TF‑IDF**  
  - Accuracy ~0.801, slightly higher overall than BOW.  
  - Balanced precision and recall across both classes.  
  - Down‑weighting common words helped Logistic Regression separate classes more effectively.

- **Overall takeaway**  
  - Logistic Regression improved both models compared to baseline.  
  - TF‑IDF edges out BOW in accuracy and balance, showing that weighting rare words adds discriminative power.  
  - For short texts like tweets, BOW is still competitive, but TF‑IDF + Logistic Regression is the stronger choice here.  
  - Next step: experiment with **n‑grams (bigrams/trigrams)** or embeddings (Word2Vec, GloVe) to capture richer context and potentially push accuracy beyond 0.80.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

lr = LogisticRegression(max_iter=500, solver='lbfgs', n_jobs=-1)

param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l2','l1'] 
}

grid_search_lr = GridSearchCV(
    estimator=lr, 
    param_grid=param_grid, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1,
    verbose=2 
)

print("Tuning Logistic Regression... This will take a few minutes.")
grid_search_lr.fit(X_train_bow, y_train)

print("\n--- Tuning Complete ---")
print("Best Parameters Found:", grid_search_lr.best_params_)
print("Best Cross-Validation Accuracy:", grid_search_lr.best_score_)

Tuning Logistic Regression... This will take a few minutes.
Fitting 3 folds for each of 4 candidates, totalling 12 fits

--- Tuning Complete ---
Best Parameters Found: {'C': 1.0, 'penalty': 'l2'}
Best Cross-Validation Accuracy: 0.7956428058793588


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

lr = LogisticRegression(max_iter=500, solver='lbfgs', n_jobs=-1)
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l2','l1'] 
}
grid_search_lr = GridSearchCV(
    estimator=lr, 
    param_grid=param_grid, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1,
    verbose=2  
)

print("Tuning Logistic Regression... This will take a few minutes.")
grid_search_lr.fit(X_train_tfidf, y_train)


print("\n--- Tuning Complete ---")
print("Best Parameters Found:", grid_search_lr.best_params_)
print("Best Cross-Validation Accuracy:", grid_search_lr.best_score_)

Tuning Logistic Regression... This will take a few minutes.
Fitting 3 folds for each of 4 candidates, totalling 12 fits



--- Tuning Complete ---
Best Parameters Found: {'C': 1.0, 'penalty': 'l2'}
Best Cross-Validation Accuracy: 0.7915303215689251


In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

lr_model = LogisticRegression(max_iter=1000, n_jobs=-1,C= 1.0, penalty= 'l2')

print("Training Logistic Regression...")
lr_model.fit(X_train_bow, y_train)

y_pred_lr = lr_model.predict(X_test_bow)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))


Training Logistic Regression...
Logistic Regression Accuracy: 0.8007170543861091

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.78      0.80    157544
           1       0.79      0.82      0.81    158750

    accuracy                           0.80    316294
   macro avg       0.80      0.80      0.80    316294
weighted avg       0.80      0.80      0.80    316294



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

lr_model = LogisticRegression(max_iter=1000, n_jobs=-1,C= 1.0, penalty= 'l2')

print("Training Logistic Regression...")
lr_model.fit(X_train_tfidf, y_train)

y_pred_tfidlr = lr_model.predict(X_test_tfidf)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_tfidlr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tfidlr))
print("\nconfusion_matrix TF-IDF:\n")
print(confusion_matrix(y_test, y_pred_tfidlr))

Training Logistic Regression...
Logistic Regression Accuracy: 0.7953296616439136

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.77      0.79    157544
           1       0.78      0.82      0.80    158750

    accuracy                           0.80    316294
   macro avg       0.80      0.80      0.80    316294
weighted avg       0.80      0.80      0.80    316294


confusion_matrix TF-IDF:

[[122066  35478]
 [ 29258 129492]]


### Insights: Logistic Regression with TF‑IDF (after tuning)

- Achieved ~0.80 accuracy, outperforming baseline models.  
- Confusion matrix shows balanced predictions: ~122k correct negatives, ~129k correct positives.  
- Slight bias toward predicting positive sentiment (more false positives than false negatives).  
- Precision and recall are well balanced across both classes, confirming TF‑IDF + Logistic Regression as a strong choice.